## **MoE**

| Expert | Modelo | Provider | Self-label | Inteligência |
|--------|--------|----------|------------|-------------|
| Expert 1 | **GPT-4** | iaedu.pt (4 contas) | OpenAI | Frontier |
| Expert 2 | **Kimi K2 0905** | Groq | — (neutro!) | #2 no Groq |
| Expert 3 | **Llama 4 Scout** | Groq | Meta | MoE 109B |

**Mudanças vs versão anterior:**
- Kimi K2 0905 (Moonshot AI) substitui Llama 3.3 70B — mais inteligente e **totalmente neutro** (não pertence a nenhuma das 5 classes)
- Llama 4 Scout substitui Mistral Small — modelo mais recente, mais capaz
- Tudo via Groq (exceto GPT-4) — menos APIs, menos falhas, código mais simples


In [9]:
import pandas as pd
import numpy as np
import time
import json
import re
import requests
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from groq import Groq

sns.set_style('whitegrid')
LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']


### **1. Keys + Clientes**

In [10]:
# Groq — Kimi K2 + Llama 4 Scout (mesmas keys para ambos)
GROQ_KEYS = [
    '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', 
    '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',
    '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',
    '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',
]

groq_clients = [Groq(api_key=key) for key in GROQ_KEYS if key]
groq_idx = 0 

# Modelos do Groq
KIMI_MODEL  = 'moonshotai/kimi-k2-instruct-0905'   # Neutro, #2 inteligência no Groq, 262K contexto
SCOUT_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'  # MoE 109B, multimodal

# iaedu.pt (GPT-4)
CONTAS_IAEDU = [
    {"nome": "Conta 1", "api_key": "sk-usr-5gned314t8prpi6cakj0v342vreij3mzh7x", "endpoint": "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream", "channel_id": "cmmuw75o1atqyhv015ja9fmo2"},
    {"nome": "Conta 2", "api_key": "sk-usr-dq0sqm5wqdbxtkk2tez3oqr7p726zrfhk5u", "endpoint": "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream", "channel_id": "cmmytnq8rhus4hv01e3yjj881"},
    {"nome": "Conta 3", "api_key": "sk-usr-23gdi3yjieky9p4prsprwk4fattnmiwtdg5", "endpoint": "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream", "channel_id": "cmmytlxmdhunehv01w2ns6sdp"},
    {"nome": "Conta 4", "api_key": "sk-usr-4wm81k1mxprmejf3ywwykcq2k9667xpnsbv", "endpoint": "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream", "channel_id": "cmmz16ptfigwjhv01dckivs9u"},
]
iaedu_idx = 0


### **2. Dados**

In [11]:
df_support = pd.read_csv('../database/dataset-subm1-labels.csv', sep=';')
df_support.columns = df_support.columns.str.strip().str.lower()

df_test = pd.read_csv('../database/dataset-samples.csv', sep=';')
df_test.columns = df_test.columns.str.strip().str.lower()

N_PER_CLASS = 2
support_set = pd.concat([
    group.sample(min(N_PER_CLASS, len(group)), random_state=42)
    for _, group in df_support.groupby('label')
]).reset_index(drop=True)

print(f'Few-shot: {len(support_set)} | Teste: {len(df_test)}')
print(f'Distribuição support: {dict(support_set["label"].value_counts())}')


Few-shot: 10 | Teste: 125
Distribuição support: {'Anthropic': np.int64(2), 'Google': np.int64(2), 'Human': np.int64(2), 'Meta': np.int64(2), 'OpenAI': np.int64(2)}


### **3. Prompt**

In [12]:
TRUNC_EXAMPLES = 600   # mais texto nos exemplos = mais padrões visíveis
TRUNC_QUERY    = 1200  # mais texto na query = mais sinal

STYLE_HINTS = """
DETAILED STYLISTIC FINGERPRINTS — use these to distinguish the sources:

HUMAN writing:
- Irregular paragraph lengths, some very short, some long
- Personal opinions, anecdotes, or first-person references
- Informal transitions ("So," "Well," "Anyway," "Now,")
- Minor imperfections: typos, run-on sentences, inconsistent formatting
- Citations, references, or links to external sources
- Wikipedia-style neutral encyclopedic tone when factual
- Does NOT use formulaic introductions or conclusions

ANTHROPIC (Claude) writing:
- Frequent hedging: "it's worth noting", "it's important to consider", "arguably"
- Balanced/nuanced: presents multiple perspectives, avoids taking strong positions
- Uses phrases like "I think", "I'd say", "I should note"
- Empathetic and careful tone, avoids being overly assertive
- Often acknowledges limitations or complexity
- Tends to use em-dashes (—) frequently
- Moderate length, well-structured but not overly formatted

OPENAI (GPT) writing:
- Confident and assertive tone, makes definitive statements
- Heavy use of bullet points, numbered lists, and bold formatting
- Formulaic structure: clear intro paragraph, body with headers, wrap-up conclusion
- Uses phrases like "Certainly!", "Absolutely!", "Great question!"
- Comprehensive coverage — tries to be thorough and complete
- Often starts responses with a direct answer then elaborates
- Uses transitional phrases like "Moreover," "Furthermore," "In addition,"

GOOGLE (Gemini) writing:
- Concise and direct, gets to the point quickly
- Shorter paragraphs and shorter overall responses
- Technical precision, uses specific terminology
- Less decorative language than GPT — more functional
- Often uses "Here's" or "Here are" to introduce content
- May include asterisks (*) for emphasis instead of bold
- Less likely to use numbered lists, prefers flowing prose

META (Llama) writing:
- Less polished, sometimes awkward phrasing or grammar
- Repetitive: restates the same idea in slightly different words
- Simpler vocabulary, shorter sentences
- May include unnecessary filler phrases
- Sometimes produces slightly off-topic tangents
- Less structured than GPT/Claude — more stream-of-consciousness
- Can be overly verbose without adding substance
"""

def build_prompt(text, support_examples):
    examples_block = ''
    for _, row in support_examples.iterrows():
        ex_text = row['text'][:TRUNC_EXAMPLES]
        examples_block += f'Text: {ex_text}\nCategory: {row["label"]}\n\n'
    return f"""You are a forensic linguist specialized in AI-generated text detection.
Your job is to classify text into ONE of 5 categories based on WRITING STYLE, not content.

Categories:
- Human (written by a real person)
- Anthropic (generated by Claude)
- Google (generated by Gemini)
- Meta (generated by Llama)
- OpenAI (generated by GPT)

CRITICAL: Do NOT default to OpenAI just because text looks AI-generated. Each AI has a DISTINCT style.
Pay close attention to the differences below:
{STYLE_HINTS}

Here are {len(support_examples)} labeled examples for reference:

{examples_block}
Now analyze this text step by step:
1. First, decide: is this Human or AI? (look for imperfections, personal voice, citations)
2. If AI, examine the SPECIFIC style markers to distinguish which AI.
3. Output your final answer as a SINGLE word on the last line: Human, Anthropic, Google, Meta, or OpenAI

Text: {text[:TRUNC_QUERY]}

Analysis and answer:"""


### **4. Expert Wrappers**

- **GPT-4** via iaedu.pt — 4 contas em rotação
- **Kimi K2 0905** via Groq — retry infinito, totalmente neutro
- **Llama 4 Scout** via Groq — retry infinito


In [13]:
def ask_gpt4(prompt):
    """Expert 1: GPT-4 via iaedu.pt"""
    global iaedu_idx
    attempt = 0
    while True:
        conta = CONTAS_IAEDU[iaedu_idx % len(CONTAS_IAEDU)]
        iaedu_idx += 1
        try:
            headers = {"x-api-key": conta["api_key"]}
            payload = {
                "channel_id": (None, conta["channel_id"]),
                "message": (None, prompt),
                "thread_id": (None, conta["channel_id"]),
                "user_info": (None, json.dumps({"name": "student"})),
            }
            resp = requests.post(
                conta["endpoint"], headers=headers,
                files=payload, stream=True, timeout=(10, 30)
            )
            
            if resp.status_code == 429: raise ValueError("HTTP 429: Rate limit atingido.")
            resp.raise_for_status()

            full_text = ""
            for line in resp.iter_lines():
                if not line: continue
                try:
                    parsed = json.loads(line.decode())
                    if isinstance(parsed, dict) and isinstance(parsed.get("content"), str):
                        full_text += parsed["content"]
                    elif isinstance(parsed, str):
                        full_text += parsed
                except (json.JSONDecodeError, UnicodeDecodeError):
                    pass

            result = full_text.strip()
            if "Rate limit" in result or "429" in result: raise ValueError(f"Rate limit detetado. O servidor disse: {result}")
            result = result.replace('Processing', '')
            result = re.sub(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', '', result).strip()

            if result: return result
            raise ValueError("Resposta vazia.")

        except KeyboardInterrupt: raise
        except Exception as e:
            attempt += 1; wait = min(10, 60)
            print(f'    GPT-4 ({conta["nome"]}): erro (tentativa {attempt}), espera {wait}s... [{type(e).__name__}]')
            time.sleep(wait)


def ask_kimi(prompt):
    """Expert 2: Kimi K2 0905 via Groq (Moonshot AI — neutro!)"""
    global groq_idx
    attempt = 0
    while True:
        current_idx = groq_idx % len(groq_clients)
        client = groq_clients[current_idx]
        groq_idx += 1
        try:
            response = client.chat.completions.create(
                messages=[{'role': 'user', 'content': prompt}],
                model=KIMI_MODEL,
                max_tokens=500, temperature=0.0,
            )
            return response.choices[0].message.content.strip()
        except KeyboardInterrupt: raise
        except Exception as e:
            attempt += 1; wait = min(15 * attempt, 120)
            print(f'    Kimi (Key {current_idx + 1}): erro (tentativa {attempt}), espera {wait}s... [{type(e).__name__}]')
            time.sleep(wait)


def ask_scout(prompt):
    """Expert 3: Llama 4 Scout via Groq"""
    global groq_idx
    attempt = 0
    while True:
        current_idx = groq_idx % len(groq_clients)
        client = groq_clients[current_idx]
        groq_idx += 1
        try:
            response = client.chat.completions.create(
                messages=[{'role': 'user', 'content': prompt}],
                model=SCOUT_MODEL,
                max_tokens=500, temperature=0.0,
            )
            return response.choices[0].message.content.strip()
        except KeyboardInterrupt: raise
        except Exception as e:
            attempt += 1; wait = min(15 * attempt, 120)
            print(f'    Scout (Key {current_idx + 1}): erro (tentativa {attempt}), espera {wait}s... [{type(e).__name__}]')
            time.sleep(wait)


def normalize_prediction(raw):
    """Extrai a última label mencionada."""
    if not raw or raw.startswith('Error'): return None
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    found_labels = []
    for label in LABELS:
        if re.search(rf'\b{label}\b', clean, re.IGNORECASE):
            found_labels.append(label)
    if not found_labels: return None
    elif len(found_labels) == 1: return found_labels[0]
    else: return max(found_labels, key=lambda l: clean.lower().rfind(l.lower()))


### **5. Teste de Conectividade**

In [14]:
print('Testando APIs...')

r = ask_gpt4('Respond with only one word: OK')
ok = 'OK' if r and not r.startswith('Error') else 'FAIL'
print(f'  {ok} GPT-4 (iaedu):        "{r[:100]}"')

r = ask_kimi('Respond with only one word: OK')
ok = 'OK' if r and not r.startswith('Error') else 'FAIL'
print(f'  {ok} Kimi K2 0905 (Groq):  "{r[:80]}"')

r = ask_scout('Respond with only one word: OK')
ok = 'OK' if r and not r.startswith('Error') else 'FAIL'
print(f'  {ok} Llama 4 Scout (Groq): "{r[:80]}"')


Testando APIs...
    GPT-4 (Conta 1): erro (tentativa 1), espera 10s... [ValueError]
    GPT-4 (Conta 2): erro (tentativa 2), espera 10s... [ValueError]
    GPT-4 (Conta 3): erro (tentativa 3), espera 10s... [ValueError]
    GPT-4 (Conta 4): erro (tentativa 4), espera 10s... [ValueError]
    GPT-4 (Conta 1): erro (tentativa 5), espera 10s... [ValueError]


KeyboardInterrupt: 

### **6. MoE Gating**

- **GPT-4** da OpenAI → penalizado quando vota "OpenAI"
- **Kimi K2** da Moonshot → **totalmente neutro** (sem self-label em nenhuma das 5 classes)
- **Llama 4 Scout** da Meta → penalizado quando vota "Meta"


In [ ]:
# Self-labels: quem tem viés sobre quem
SELF_LABEL = {'gpt4': 'OpenAI', 'kimi': None, 'scout': 'Meta'}

# Pesos
SELF_PENALTY    = 0.25   # Penalização forte quando vota na sua própria família
CROSS_WEIGHT    = 1.0    # Peso normal
OPENAI_BOOST    = 1.0    # Sem boost OpenAI — o viés já é enorme naturalmente

# Multiplicadores de confiança por expert
# Kimi é o expert mais confiável (neutro + inteligente)
EXPERT_MULTIPLIER = {'gpt4': 0.95, 'kimi': 1.15, 'scout': 0.90}

def moe_vote(expert_preds):
    votes = Counter()
    for name, pred in expert_preds.items():
        if pred is None: continue
        if pred == SELF_LABEL.get(name): weight = SELF_PENALTY
        elif pred == 'OpenAI' and SELF_LABEL.get(name) is None: weight = OPENAI_BOOST
        else: weight = CROSS_WEIGHT
        
        weight *= EXPERT_MULTIPLIER.get(name, 1.0)
        votes[pred] += weight
        
    if not votes: return None, {}
    
    cross = Counter()
    for name, pred in expert_preds.items():
        if pred and pred != SELF_LABEL.get(name):
            cross[pred] += 1
            
    # Se dois modelos concordam na mesma etiqueta (que não a sua própria), eles ganham
    for label, count in cross.most_common():
        if count >= 2:
            return label, {'method': 'cross_consensus', 'votes': dict(votes)}
            
    return votes.most_common(1)[0][0], {'method': 'weighted_vote', 'votes': dict(votes)}


### **7. Correr o MoE**

In [ ]:
def run_moe(df_data, support_df, sleep_between=10.0):
    results = []
    hits = {'gpt4': 0, 'kimi': 0, 'scout': 0, 'moe': 0}
    total = 0

    for idx, row in tqdm(df_data.iterrows(), total=len(df_data), desc='MoE'):
        prompt = build_prompt(row['text'], support_df)
        true_label = row.get('label', None)

        raw_gpt4 = ask_gpt4(prompt)
        pred_gpt4 = normalize_prediction(raw_gpt4)

        raw_kimi = ask_kimi(prompt)
        pred_kimi = normalize_prediction(raw_kimi)

        raw_scout = ask_scout(prompt)
        pred_scout = normalize_prediction(raw_scout)

        expert_preds = {'gpt4': pred_gpt4, 'kimi': pred_kimi, 'scout': pred_scout}
        final_pred, vote_info = moe_vote(expert_preds)

        results.append({
            'id': row.get('id', idx), 'true_label': true_label,
            'gpt4_raw': raw_gpt4, 'gpt4_pred': pred_gpt4,
            'kimi_raw': raw_kimi, 'kimi_pred': pred_kimi,
            'scout_raw': raw_scout, 'scout_pred': pred_scout,
            'moe_pred': final_pred, 'vote_info': json.dumps(vote_info),
        })

        if true_label:
            total += 1
            if pred_gpt4 == true_label:  hits['gpt4'] += 1
            if pred_kimi == true_label:  hits['kimi'] += 1
            if pred_scout == true_label: hits['scout'] += 1
            if final_pred == true_label: hits['moe'] += 1

            ok = 'Y' if final_pred == true_label else 'X'
            print(
                f'  [{ok}] #{total:3d} | Real={true_label[:7]:7s} '
                f'| GPT4={str(pred_gpt4)[:7]:7s} Kimi={str(pred_kimi)[:7]:7s} Scout={str(pred_scout)[:7]:7s} '
                f'-> MoE={str(final_pred)[:7]:7s} '
                f'| Accs MoE={hits["moe"]/total:.0%}'
            )

        time.sleep(sleep_between)

    return pd.DataFrame(results)

print('MoE v7 (GPT-4 + Kimi K2 + Llama 4 Scout) — prompt melhorado + CoT')
df_results = run_moe(df_test, support_set, sleep_between=10.0)
df_results.to_csv('moe_results_v7.csv', index=False, sep=';')
print('Done!')


MoE v7 (GPT-4 + Kimi K2 + Llama 4 Scout) — prompt melhorado + CoT


MoE:   0%|          | 0/125 [00:00<?, ?it/s]

    GPT-4 (Conta 2): erro (tentativa 1), espera 10s... [ValueError]
  [X] #  1 | Real=Human   | GPT4=Meta    Kimi=Meta    Scout=Anthrop -> MoE=Meta    | Accs MoE=0%
  [Y] #  2 | Real=Meta    | GPT4=OpenAI  Kimi=Meta    Scout=Google  -> MoE=Meta    | Accs MoE=50%
    GPT-4 (Conta 1): erro (tentativa 1), espera 10s... [ValueError]
    GPT-4 (Conta 2): erro (tentativa 2), espera 10s... [ValueError]
    GPT-4 (Conta 3): erro (tentativa 3), espera 10s... [ValueError]
  [X] #  3 | Real=Google  | GPT4=OpenAI  Kimi=OpenAI  Scout=Google  -> MoE=OpenAI  | Accs MoE=33%
    GPT-4 (Conta 1): erro (tentativa 1), espera 10s... [ValueError]
    GPT-4 (Conta 2): erro (tentativa 2), espera 10s... [ValueError]
    GPT-4 (Conta 3): erro (tentativa 3), espera 10s... [ValueError]


KeyboardInterrupt: 

### **8. Avaliacao**

In [ ]:
def evaluate_model(df_res, pred_col, title=''):
    valid = df_res.dropna(subset=[pred_col, 'true_label'])
    preds, golds = valid[pred_col].tolist(), valid['true_label'].tolist()
    if not preds: return 0.0
    acc = sum(p == g for p, g in zip(preds, golds)) / len(preds)
    print(f'\n{"="*60}\n{title} - Accuracy: {acc:.2%} ({len(preds)}/{len(df_res)})\n{"="*60}')
    print(classification_report(golds, preds, labels=LABELS, zero_division=0))
    cm = confusion_matrix(golds, preds, labels=LABELS)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel('Previsao'); ax.set_ylabel('Real'); ax.set_title(f'{title} - {acc:.2%}')
    plt.tight_layout(); plt.show()
    return acc


accs = {}
accs['GPT-4']          = evaluate_model(df_results, 'gpt4_pred',   'GPT-4 (iaedu)')
accs['Kimi K2 0905']   = evaluate_model(df_results, 'kimi_pred',   'Kimi K2 0905 (Groq)')
accs['Llama 4 Scout']  = evaluate_model(df_results, 'scout_pred',  'Llama 4 Scout (Groq)')
accs['MoE Ensemble']   = evaluate_model(df_results, 'moe_pred',    'MoE Ensemble')

print('\nRESUMO:')
for name, acc in sorted(accs.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:20s}: {acc:.2%}')


### **9. Analise de Erros**

In [ ]:
errors = df_results[df_results['moe_pred'] != df_results['true_label']]
print(f'Erros: {len(errors)}/{len(df_results)}\n')

for _, row in errors.iterrows():
    print(f'[{row["id"]}] Real={row["true_label"]} -> MoE={row["moe_pred"]}  |  '
          f'GPT4={row["gpt4_pred"]}  Kimi={row["kimi_pred"]}  Scout={row["scout_pred"]}')
